In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Base sem quali

In [ ]:
df_sem_quali = pd.read_csv(r"D:\pedro\Documents\modelo_f1\DATA\oficial.csv")
df_sem_quali.drop(columns=["posicao_quali_atual", "q1_atual", "q2_atual", "q3_atual", "dif_para_pole_atual"], inplace=True)
df_sem_quali.head()

In [ ]:
df_sem_quali["target"] = (df_sem_quali["target"] <= 3).astype(int)

In [ ]:
df_sem_quali["posicao_equipe_anterior"] = pd.to_numeric(df_sem_quali["posicao_equipe_anterior"], errors="coerce")

df_sem_quali.info()

In [ ]:
df_sem_quali.describe()

In [ ]:
df_sem_quali.drop(columns=["id_piloto_atual"], inplace=True)

In [ ]:
X = df_sem_quali.drop(columns=['target'])
y = df_sem_quali['target']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
onehotencoder = ColumnTransformer(transformers=[
    ('OneHot', OneHotEncoder(handle_unknown='ignore'), ['id_circuito_atual', 'id_equipe_atual', 'status'])
], remainder='passthrough')

In [ ]:
X_train_transformed = onehotencoder.fit_transform(X_train)
X_test_transformed = onehotencoder.transform(X_test)

In [ ]:
param_grid = {
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.03, 0.05],
    "n_estimators": [300, 500, 700, 1000],
    "scale_pos_weight": [2, 3],       
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.6, 0.7, 0.8], 
    "min_child_weight": [5, 7, 10],     
    "gamma": [0.2, 0.4, 0.6],          
    "reg_alpha": [0, 0.1, 0.5],       
    "reg_lambda": [1, 1.5, 2],       
}

xgb_model = xgb.XGBClassifier(random_state=42)

grid_search = RandomizedSearchCV(xgb_model, param_grid, cv=10, scoring="accuracy", n_iter=100, n_jobs=-1, verbose=2, random_state=42)
grid_search.fit(X_train_transformed, y_train)

best_xgb = grid_search.best_estimator_

print("Melhores parâmetros:", grid_search.best_params_)
print("Melhor acurácia:", grid_search.best_score_)

In [ ]:
y_pred = best_xgb.predict(X_test_transformed)

test_accuracy = accuracy_score(y_test, y_pred)
print("Acurácia: ", test_accuracy)

print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nMatriz de confusão:\n", confusion_matrix(y_test, y_pred))

Entendendo as features utilizadas

In [ ]:
colunas_ohe = onehotencoder.named_transformers_['OneHot'].get_feature_names_out(['id_circuito_atual', 'id_equipe_atual', 'status'])
colunas_resto = [c for c in X.columns if c not in ['id_circuito_atual', 'id_equipe_atual']]
feature_names = list(colunas_ohe) + list(colunas_resto)

feature_importances = best_xgb.feature_importances_
sorted_indices = np.argsort(feature_importances)[::-1]

plt.figure(figsize=(14, 5))
plt.bar(range(len(feature_importances)), feature_importances[sorted_indices], align="center")
plt.xticks(range(len(feature_importances)), np.array(feature_names)[sorted_indices], rotation=90)
plt.xlabel("Importância da Feature")
plt.title("XGB Importância da Feature para o DF da F1")
plt.tight_layout()
plt.show()

In [ ]:
import joblib

joblib.dump(best_xgb, r"D:\pedro\Documents\modelo_f1\modelos\modelo_sem_quali.pkl")
joblib.dump(onehotencoder, r"D:\pedro\Documents\modelo_f1\modelos\encoder_sem_quali.pkl")

# Testes

In [3]:
import joblib
import pandas as pd

modelo = joblib.load(r"D:\pedro\Documents\modelo_f1\modelos\modelo_sem_quali.pkl")
encoder = joblib.load(r"D:\pedro\Documents\modelo_f1\modelos\encoder_sem_quali.pkl")

df = pd.read_csv(r"D:\pedro\Documents\modelo_f1\DATA\a.csv")

X_train_transformed = encoder.transform(df)
predicoes = modelo.predict_proba(X_train_transformed)

print(predicoes)

[[0.53330976 0.46669024]
 [0.80553794 0.19446206]
 [0.58441913 0.41558087]
 [0.8845336  0.11546642]
 [0.5863427  0.41365728]
 [0.9749756  0.02502443]
 [0.98119414 0.01880584]
 [0.9793893  0.02061068]
 [0.9480714  0.0519286 ]
 [0.9677106  0.03228939]
 [0.99372494 0.00627504]
 [0.9933134  0.00668664]
 [0.99157286 0.00842717]
 [0.9977092  0.00229079]
 [0.80807686 0.19192317]
 [0.19950408 0.8004959 ]
 [0.9857652  0.0142348 ]
 [0.995454   0.00454597]
 [0.99650294 0.00349705]
 [0.9839064  0.01609364]
 [0.9964133  0.00358668]
 [0.9962966  0.00370341]]


In [4]:
probs = modelo.predict_proba(X_train_transformed)[:, 1]

df_result = df.copy()
df_result['prob_podio'] = probs
df_result['predicao'] = modelo.predict(X_train_transformed)

print(df_result[['id_piloto_atual', 'prob_podio', 'predicao']]
      .sort_values('prob_podio', ascending=False))

   id_piloto_atual  prob_podio  predicao
15       antonelli    0.800496         1
0         hamilton    0.466690         0
2           norris    0.415581         0
4          piastri    0.413657         0
1          russell    0.194462         0
14         leclerc    0.191923         0
3   max_verstappen    0.115466         0
8   arvid_lindblad    0.051929         0
9        colapinto    0.032289         0
5           hadjar    0.025024         0
7           lawson    0.020611         0
6            gasly    0.018806         0
19      hulkenberg    0.016094         0
16         bearman    0.014235         0
12            ocon    0.008427         0
11           sainz    0.006687         0
10       bortoleto    0.006275         0
17           albon    0.004546         0
21          stroll    0.003703         0
20          bottas    0.003587         0
18          alonso    0.003497         0
13           perez    0.002291         0


In [ ]:
import os
import streamlit as st
from google.cloud import storage
from google.oauth2 import service_account
import pickle

def salva_gcs(caminho_gcs, caminho_local):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "D:\pedro\Documents\modelo_f1\clever-axe-457319-g8-833d2d4ab67f.json"

    client = client = storage.Client()
    bucket = client.bucket("f1-dashboard-pilotos")
    blob = bucket.blob(caminho_gcs)

    blob.upload_from_filename(caminho_local)

modelos = [
    ("modelos/modelo_sem_quali.pkl",  r"D:\pedro\Documents\modelo_f1\modelos\modelo_sem_quali.pkl"),
    ("modelos/encoder_sem_quali.pkl", r"D:\pedro\Documents\modelo_f1\modelos\encoder_sem_quali.pkl"),
]
for caminho_gcs, caminho_local in modelos:
    salva_gcs(caminho_gcs, caminho_local)

In [ ]:
from sklearn.metrics import classification_report
import json
import os

metricas = classification_report(
    y_test,
    y_pred,
    output_dict=True
)

metricas_classe_1 = metricas["1"]
metricas_classe_1["accuracy"] = test_accuracy

def salva_gcs(dados):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "D:\pedro\Documents\modelo_f1\clever-axe-457319-g8-833d2d4ab67f.json"

    client = client = storage.Client()
    bucket = client.bucket("f1-dashboard-pilotos")
    blob = bucket.blob("metricas_sem_quali.json")
    blob.upload_from_string(
        json.dumps(dados), 
        content_type="application/json"
    )

salva_gcs(metricas_classe_1)